# 🛣️ RoadSign Evaluator — SEÑALIZACIÓN HORIZONTAL (versión KAGGLE)

Entrena el modelo separado de marcas viales (pasos de cebra, líneas, flechas, etc.). Versión para **Kaggle Notebooks**.

---

## ⚙️ ANTES DE EMPEZAR (en Kaggle)

1. Cuenta en kaggle.com con **teléfono verificado**.
2. Panel derecho: **Accelerator → GPU P100** e **Internet → On**.
3. **Run → Run all**.
4. API key de Roboflow en el Paso 2.

## 📥 Al terminar

`model_horizontal.onnx` y `labels_horizontal.json` quedan en /kaggle/working. Descárgalos del panel derecho (Output) y súbelos a `models/` en tu repo.

## ⚠️ Nota honesta

Las marcas viales tienen datasets pequeños y son más difíciles que las verticales. Espera precisión menor; combinamos datasets para mejorarla.

## Paso 1 — GPU + herramientas

In [ ]:
import subprocess, sys
gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print('✅ GPU activa' if 'GPU' in gpu.stdout or 'P100' in gpu.stdout else '⚠️ Activa GPU en el panel derecho')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics', 'roboflow', 'onnx', 'onnxslim', 'pyyaml'])
print('✅ Herramientas instaladas')

## Paso 2 — Descargar datasets de marcas viales

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
TU_API_KEY = "TU_API_KEY"
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os
os.chdir('/kaggle/working')
from roboflow import Roboflow
rf = Roboflow(api_key=TU_API_KEY)
datasets = []
try:
    p1 = rf.workspace('mvi-89ske').project('road-markings-o6ofy')
    d1 = p1.version(1).download('yolov8', location='/kaggle/working/ds_mvi')
    datasets.append(d1.location); print(f'✅ Road Markings MVI')
except Exception as e: print(f'⚠️ MVI: {e}')
try:
    p2 = rf.workspace('yolo-datasets-f9og9').project('crosswalks-zn9wq-kfndo')
    d2 = p2.version(1).download('yolov8', location='/kaggle/working/ds_cross')
    datasets.append(d2.location); print(f'✅ Crosswalks')
except Exception as e: print(f'⚠️ Crosswalks: {e}')
print(f'\n{len(datasets)} datasets')
if not datasets: raise RuntimeError('Sin datasets. Revisa API key e Internet ON.')

## Paso 3 — Unificar datasets

In [ ]:
import os, shutil, yaml, glob
UNIFIED_CLASSES = ['paso_peatones','linea_continua','linea_discontinua','doble_linea','carril_bus','flecha_carril','zigzag','linea_detencion']
def map_class_name(o):
    o = o.lower().replace('-',' ').replace('_',' ')
    if 'zebra' in o or 'crosswalk' in o or 'cross walk' in o or 'paso' in o or 'pedestrian cross' in o: return 0
    if 'double' in o: return 3
    if 'zigzag' in o or 'zig zag' in o: return 6
    if 'bus' in o: return 4
    if 'turn' in o or 'arrow' in o or 'flecha' in o: return 5
    if 'stop line' in o or 'stop-line' in o or 'detencion' in o or 'detention' in o: return 7
    if 'broken' in o or 'dash' in o or 'discontinu' in o: return 2
    if 'solid' in o or 'continu' in o or 'line' in o: return 1
    return None
MERGED = '/kaggle/working/merged'
for s in ['train','valid']:
    os.makedirs(f'{MERGED}/{s}/images', exist_ok=True)
    os.makedirs(f'{MERGED}/{s}/labels', exist_ok=True)
counter = 0
for ds in datasets:
    yf = os.path.join(ds, 'data.yaml')
    if not os.path.exists(yf): continue
    with open(yf) as f: cfg = yaml.safe_load(f)
    names = cfg.get('names', [])
    if isinstance(names, dict): names = [names[k] for k in sorted(names.keys())]
    remap = {}
    for i, nm in enumerate(names):
        ni = map_class_name(nm)
        if ni is not None: remap[i] = ni
    for split in ['train','valid','test']:
        idir = os.path.join(ds, split, 'images'); ldir = os.path.join(ds, split, 'labels')
        if not os.path.isdir(idir): continue
        tgt = 'valid' if split in ('valid','test') else 'train'
        for img in glob.glob(f'{idir}/*'):
            base = os.path.splitext(os.path.basename(img))[0]
            lbl = os.path.join(ldir, base + '.txt'); new = []
            if os.path.exists(lbl):
                with open(lbl) as lf:
                    for line in lf:
                        p = line.split()
                        if not p: continue
                        oc = int(p[0])
                        if oc in remap: p[0] = str(remap[oc]); new.append(' '.join(p))
            if new:
                counter += 1; ext = os.path.splitext(img)[1]
                shutil.copy(img, f'{MERGED}/{tgt}/images/img{counter}{ext}')
                with open(f'{MERGED}/{tgt}/labels/img{counter}.txt','w') as of: of.write('\n'.join(new))
with open(f'{MERGED}/data.yaml','w') as f:
    yaml.dump({'train': f'{MERGED}/train/images','val': f'{MERGED}/valid/images','nc': len(UNIFIED_CLASSES),'names': UNIFIED_CLASSES}, f)
nt = len(glob.glob(f'{MERGED}/train/images/*')); nv = len(glob.glob(f'{MERGED}/valid/images/*'))
print(f'✅ {counter} imágenes unificadas | Train: {nt} Val: {nv}')
if nt < 50: print('⚠️ Pocas imágenes; modelo limitado.')

## Paso 4 — Entrenar

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8s.pt')
model.train(
    data=f'{MERGED}/data.yaml',
    epochs=200, patience=40, imgsz=640, batch=16,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.5,
    degrees=8, translate=0.15, scale=0.5, perspective=0.0005,
    fliplr=0.5, mosaic=1.0, mixup=0.1, cos_lr=True, lr0=0.01,
    project='/kaggle/working/train', name='horizontal', exist_ok=True,
)
print('✅ Entrenamiento completado')

## Paso 5 — Métricas y exportación

In [ ]:
import shutil, json, os
os.chdir('/kaggle/working')
best = YOLO('/kaggle/working/train/horizontal/weights/best.pt')
metrics = best.val(data=f'{MERGED}/data.yaml')
print(f'📊 mAP50: {metrics.box.map50:.3f} | mAP50-95: {metrics.box.map:.3f}')
onnx_path = best.export(format='onnx', imgsz=640, opset=12, simplify=True, dynamic=False)
shutil.move(onnx_path, '/kaggle/working/model_horizontal.onnx')
names = best.names; labels = [names[i] for i in range(len(names))]
with open('/kaggle/working/labels_horizontal.json','w',encoding='utf-8') as f:
    json.dump(labels, f, ensure_ascii=False, indent=2)
size_mb = os.path.getsize('/kaggle/working/model_horizontal.onnx')/1024/1024
print(f'✅ model_horizontal.onnx ({size_mb:.1f} MB) + labels_horizontal.json')
print(f'Clases: {labels}')
print('\n📥 Descarga del panel derecho (Output) y sube a models/ en tu repo.')